# ResNet-18 custom codebook quantization accuracy from 2 to 8 bits per weight

This notebook evaluates the same custom N-bit codebook function repeatedly from 2 through 8 bits per weight. For every bit width (b), the scalar codebook contains (K=2^b) values. Gong et al. found scalar k-means surprisingly strong for CNN parameters; Choi et al. motivates the bounded gradient-second-moment sensitivity used to refine assignments and centroids. Product quantization from Jégou et al. and Gong et al. remains available in the implementation, but is not the default here because the earlier two-weight product budget was only b/2 bits per weight and measured substantially worse.

With `VECTOR_DIM=1`, one assignment represents one weight, so assignment bits and bits per weight are identical before codebook and preserved-tensor overhead. Quantized weights are reconstructed for FP32 Keras evaluation; activations remain FP32, no fake quantization is used, and no optimizer step changes the trained model.


In [ ]:
from pathlib import Path
import gc
import json
import sys
import time

import keras
import keras_hub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.quantization.custom_quantization import (
    codebook_vectorize_model_n_bits,
    estimate_gradient_importance,
    quantization_mse,
    reconstruct_codebook_model_weights,
    reconstruct_codebook_tensor,
)

np.random.seed(42)
tf.random.set_seed(42)
print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {keras.__version__}")


## 1. Configuration

`ASSIGNMENT_BIT_WIDTHS` is the sweep variable. Because `VECTOR_DIM=1`, it is also the actual bit-width per quantized weight. Codebook size is derived as `2 ** bits`; all other settings and tensor selections remain identical.


In [ ]:
MODEL_PATH = (
    PROJECT_ROOT / "artifacts" / "resnet18_cifar10_training"
    / "resnet18_cifar10_fp32.keras"
)
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "resnet18_custom_codebook_bit_sweep"

ASSIGNMENT_BIT_WIDTHS = list(range(2, 9))
VECTOR_DIM = 1  # True N-bit-per-weight scalar codebook.
CODEBOOK_DTYPE = np.float32
KMEANS_ITERATIONS = 30
MAX_KMEANS_SAMPLES = 100_000
QUANTIZE_MIN_RANK = 2
RANDOM_SEED = 42
CODEBOOK_MODE = "vector"  # One learned scalar codebook per tensor.
NUM_IMPORTANCE_CALIBRATION_SAMPLES = 256
CALIBRATION_BATCH_SIZE = 16
PRESERVE_FIRST_AND_LAST_KERNELS = True
NUM_EVALUATION_SAMPLES = None  # None evaluates all 10,000 test images.
BATCH_SIZE = 32

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Missing trained model: {MODEL_PATH}. Run notebook 06 first."
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Assignment bit widths: {ASSIGNMENT_BIT_WIDTHS}")
print(f"Codebook sizes: {[2 ** bits for bits in ASSIGNMENT_BIT_WIDTHS]}")
print(f"Vector dimension: {VECTOR_DIM}")


## 2. Load the trained ResNet-18 and CIFAR-10


In [ ]:
model = keras.models.load_model(MODEL_PATH, compile=False)
(train_images, train_labels), (test_images, test_labels) = keras.datasets.cifar10.load_data()
train_labels = train_labels.reshape(-1).astype(np.int32)
test_labels = test_labels.reshape(-1).astype(np.int32)
if NUM_EVALUATION_SAMPLES is not None:
    test_images = test_images[:NUM_EVALUATION_SAMPLES]
    test_labels = test_labels[:NUM_EVALUATION_SAMPLES]

image_converter = keras_hub.layers.ResNetImageConverter(
    image_size=(224, 224),
    scale=[0.017124753831663668, 0.01750700280112045, 0.017429193899782133],
    offset=[-2.1179039301310043, -2.0357142857142856, -1.8044444444444445],
    interpolation="bicubic",
    crop_to_aspect_ratio=True,
)

def preprocess_images(images):
    return image_converter(images)

print(f"Evaluation images: {len(test_images):,}")
print(f"Sensitivity-calibration images: {NUM_IMPORTANCE_CALIBRATION_SAMPLES:,}")
print(f"Parameter tensors: {len(model.weights)}")
model.summary(expand_nested=True)


## 3. Define the mixed-precision policy and calibrate sensitivity

Floating tensors with rank 2 or higher are eligible. The first convolution kernel and final classifier kernel remain FP32 because input/output layers are commonly more sensitive to aggressive compression. Rank-1 biases and normalization parameters are already preserved by `quantize_min_rank=2`.


In [ ]:
def tensor_name(weight):
    return getattr(weight, "path", weight.name)

eligible_kernel_names = [
    tensor_name(weight)
    for weight in model.weights
    if np.issubdtype(weight.numpy().dtype, np.floating)
    and weight.numpy().ndim >= QUANTIZE_MIN_RANK
]
PRESERVED_TENSOR_NAMES = (
    {eligible_kernel_names[0], eligible_kernel_names[-1]}
    if PRESERVE_FIRST_AND_LAST_KERNELS and eligible_kernel_names
    else set()
)
selected_names = set(eligible_kernel_names) - PRESERVED_TENSOR_NAMES

selection_rows = []
for weight in model.weights:
    values = weight.numpy()
    name = tensor_name(weight)
    selection_rows.append({
        "tensor": name,
        "rank": values.ndim,
        "shape": tuple(values.shape),
        "values": values.size,
        "codebook_selected": name in selected_names,
        "preserved_sensitive": name in PRESERVED_TENSOR_NAMES,
    })
selection_details = pd.DataFrame(selection_rows)
print(f"Codebook-quantized tensors per run: {len(selected_names)}")
print("Preserved sensitive tensors:")
for name in sorted(PRESERVED_TENSOR_NAMES):
    print(f"  - {name}")
display(selection_details.groupby("rank", as_index=False).agg(
    total_tensors=("tensor", "count"),
    total_values=("values", "sum"),
    quantized_tensors=("codebook_selected", "sum"),
    preserved_sensitive_tensors=("preserved_sensitive", "sum"),
))

def sensitivity_calibration_batches():
    count = min(NUM_IMPORTANCE_CALIBRATION_SAMPLES, len(train_images))
    for start in range(0, count, CALIBRATION_BATCH_SIZE):
        stop = min(start + CALIBRATION_BATCH_SIZE, count)
        yield (
            preprocess_images(train_images[start:stop]),
            train_labels[start:stop],
        )

print("Estimating gradient/Hessian-proxy weight sensitivity...")
importance_start = time.perf_counter()
gradient_importance = estimate_gradient_importance(
    model, sensitivity_calibration_batches(), use_square_root=True
)
importance_seconds = time.perf_counter() - importance_start
missing_importance = selected_names - gradient_importance.keys()
if missing_importance:
    raise RuntimeError(f"Missing sensitivity for: {sorted(missing_importance)}")
print(f"Sensitivity estimation: {importance_seconds:.2f} s")


## 4. Prediction helper


In [ ]:
def predict_keras_labels(keras_model, raw_images, batch_size=BATCH_SIZE):
    predictions = []
    for start in range(0, len(raw_images), batch_size):
        batch = preprocess_images(raw_images[start:start + batch_size])
        logits = keras_model(batch, training=False).numpy()
        predictions.append(np.argmax(logits, axis=1))
    return np.concatenate(predictions)


## 5. Evaluate FP32 once


In [ ]:
print("Evaluating FP32 baseline...")
start = time.perf_counter()
fp32_predictions = predict_keras_labels(model, test_images)
fp32_evaluation_seconds = time.perf_counter() - start
fp32_correct = int(np.sum(fp32_predictions == test_labels))
fp32_accuracy = float(np.mean(fp32_predictions == test_labels))
fp32_parameter_bytes = int(sum(weight.numpy().nbytes for weight in model.weights))
print(f"FP32 accuracy: {fp32_accuracy * 100:.2f}%")
print(f"FP32 evaluation: {fp32_evaluation_seconds:.2f} s")


## 6. Call the same custom codebook function for 2–8 bits per weight

For each run, `K = 2 ** num_bits`. A tensor-specific scalar codebook is initialized with k-means++ and refined with bounded gradient-second-moment importance in both nearest-codeword assignment and centroid updates. No optimizer step changes the trained model.


In [ ]:
sweep_rows = []
layer_error_rows = []
original_by_name = {tensor_name(weight): weight.numpy() for weight in model.weights}

for assignment_bits in ASSIGNMENT_BIT_WIDTHS:
    codebook_size = 2 ** assignment_bits
    effective_bits_per_weight = assignment_bits / VECTOR_DIM
    print(
        f"\nEvaluating {assignment_bits}-bit weights: "
        f"K={codebook_size}, {effective_bits_per_weight:.2f} "
        "bits/weight"
    )
    quantization_start = time.perf_counter()

    # This is the one custom codebook function used for every bit width.
    codebook_result = codebook_vectorize_model_n_bits(
        model,
        num_bits=assignment_bits,
        vector_dim=VECTOR_DIM,
        mode=CODEBOOK_MODE,
        quantize_min_rank=QUANTIZE_MIN_RANK,
        excluded_tensor_names=PRESERVED_TENSOR_NAMES,
        importance_by_name=gradient_importance,
        codebook_dtype=CODEBOOK_DTYPE,
        max_kmeans_samples=MAX_KMEANS_SAMPLES,
        kmeans_iterations=KMEANS_ITERATIONS,
        seed=RANDOM_SEED,
    )
    quantization_seconds = time.perf_counter() - quantization_start

    quantized_model = keras.models.load_model(MODEL_PATH, compile=False)
    quantized_model.set_weights(
        reconstruct_codebook_model_weights(model, codebook_result)
    )
    evaluation_start = time.perf_counter()
    predictions = predict_keras_labels(quantized_model, test_images)
    evaluation_seconds = time.perf_counter() - evaluation_start

    correct = int(np.sum(predictions == test_labels))
    accuracy = float(np.mean(predictions == test_labels))
    tensor_mses = [
        quantization_mse(original_by_name[tensor.name], tensor)
        for tensor in codebook_result.tensors
    ]
    weighted_distortions = []
    for tensor, mse in zip(codebook_result.tensors, tensor_mses):
        original = original_by_name[tensor.name].astype(np.float32)
        reconstructed = reconstruct_codebook_tensor(tensor).astype(np.float32)
        importance = gradient_importance[tensor.name]
        weighted_distortion = float(
            np.sum(importance * (original - reconstructed) ** 2)
            / np.sum(importance)
        )
        weighted_distortions.append(weighted_distortion)
        layer_error_rows.append({
            "assignment_bits": assignment_bits,
            "codebook_size": codebook_size,
            "tensor": tensor.name,
            "mse": mse,
            "sensitivity_weighted_distortion": weighted_distortion,
        })

    sweep_rows.append({
        "assignment_bits": assignment_bits,
        "codebook_size_k": codebook_size,
        "vector_dim": VECTOR_DIM,
        "effective_assignment_bits_per_weight": effective_bits_per_weight,
        "quantized_tensors": len(codebook_result.tensors),
        "correct_predictions": correct,
        "test_images": len(test_labels),
        "accuracy_percent": accuracy * 100,
        "drop_from_fp32_percentage_points": (fp32_accuracy - accuracy) * 100,
        "mean_tensor_mse": float(np.mean(tensor_mses)),
        "mean_sensitivity_weighted_distortion": float(np.mean(weighted_distortions)),
        "estimated_packed_parameter_bytes": codebook_result.estimated_compressed_size_bytes,
        "estimated_packed_parameter_mib": codebook_result.estimated_compressed_size_bytes / 1024**2,
        "compression_ratio_vs_fp32": codebook_result.compression_ratio,
        "memory_reduction_percent": codebook_result.memory_reduction_percent,
        "quantization_seconds": quantization_seconds,
        "evaluation_seconds": evaluation_seconds,
    })
    print(
        f"Accuracy: {accuracy * 100:.2f}% | "
        f"drop: {(fp32_accuracy - accuracy) * 100:.2f} pp | "
        f"compression: {codebook_result.compression_ratio:.2f}x"
    )

    del predictions, quantized_model, codebook_result
    gc.collect()

sweep_table = pd.DataFrame(sweep_rows)
layer_error_table = pd.DataFrame(layer_error_rows)
sweep_table


## 7. Accuracy versus codebook weight bits

The lower annotation on each point shows K. Since one assignment represents one weight, the x-axis is the actual codebook index width per quantized weight.


In [ ]:
figure, axis = plt.subplots(figsize=(9.5, 5.8))
axis.plot(
    sweep_table["assignment_bits"],
    sweep_table["accuracy_percent"],
    marker="o",
    linewidth=2,
    markersize=7,
    label="Custom codebook VQ",
)
axis.axhline(
    fp32_accuracy * 100,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label=f"FP32 baseline ({fp32_accuracy * 100:.2f}%)",
)
for _, row in sweep_table.iterrows():
    axis.annotate(
        f"{row['accuracy_percent']:.2f}%\nK={int(row['codebook_size_k'])}",
        (row["assignment_bits"], row["accuracy_percent"]),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center",
        fontsize=8.5,
    )
axis.set_xticks(ASSIGNMENT_BIT_WIDTHS)
axis.set_xlabel("Codebook bits per quantized weight")
axis.set_ylabel("CIFAR-10 accuracy (%)")
axis.set_title("ResNet-18 sensitivity-refined codebook accuracy versus bits")
axis.grid(True, alpha=0.3)
axis.legend()
figure.tight_layout()
figure


## 8. Accuracy, codebook size, and memory summary


In [ ]:
fp32_row = pd.DataFrame([{
    "assignment_bits": "FP32",
    "codebook_size_k": 0,
    "vector_dim": 0,
    "effective_assignment_bits_per_weight": 32.0,
    "quantized_tensors": 0,
    "correct_predictions": fp32_correct,
    "test_images": len(test_labels),
    "accuracy_percent": fp32_accuracy * 100,
    "drop_from_fp32_percentage_points": 0.0,
    "estimated_packed_parameter_mib": fp32_parameter_bytes / 1024**2,
    "compression_ratio_vs_fp32": 1.0,
    "memory_reduction_percent": 0.0,
}])
final_table = pd.concat([fp32_row, sweep_table], ignore_index=True)
display_columns = [
    "assignment_bits", "codebook_size_k",
    "effective_assignment_bits_per_weight", "quantized_tensors",
    "correct_predictions", "test_images", "accuracy_percent",
    "drop_from_fp32_percentage_points", "estimated_packed_parameter_mib",
    "compression_ratio_vs_fp32", "memory_reduction_percent",
]
final_table[display_columns]


## 9. Save the experiment


In [ ]:
accuracy_path = OUTPUT_DIR / "custom_codebook_2_to_8_bit_accuracy.csv"
layer_error_path = OUTPUT_DIR / "custom_codebook_layer_mse_by_bits.csv"
selection_path = OUTPUT_DIR / "custom_codebook_tensor_selection.csv"
graph_path = OUTPUT_DIR / "custom_codebook_accuracy_vs_bits.png"
results_path = OUTPUT_DIR / "results.json"

sweep_table.to_csv(accuracy_path, index=False)
layer_error_table.to_csv(layer_error_path, index=False)
selection_details.to_csv(selection_path, index=False)
figure.savefig(graph_path, dpi=160, bbox_inches="tight")
results_path.write_text(json.dumps({
    "model": str(MODEL_PATH),
    "dataset": "CIFAR-10 test",
    "evaluation_images": int(len(test_labels)),
    "assignment_bit_widths": ASSIGNMENT_BIT_WIDTHS,
    "codebook_sizes": [2 ** bits for bits in ASSIGNMENT_BIT_WIDTHS],
    "vector_dim": VECTOR_DIM,
    "codebook_mode": CODEBOOK_MODE,
    "importance_calibration_images": NUM_IMPORTANCE_CALIBRATION_SAMPLES,
    "importance_estimation_seconds": importance_seconds,
    "importance_method": "sqrt gradient second moment / empirical Fisher proxy",
    "research_basis": [
        "Gong et al., Compressing Deep Convolutional Networks Using Vector Quantization (arXiv:1412.6115)",
        "Choi et al., Towards the Limit of Network Quantization (arXiv:1612.01543)",
        "Jegou et al., Product Quantization for Nearest Neighbor Search",
    ],
    "effective_assignment_bits_per_weight": {
        str(bits): bits / VECTOR_DIM for bits in ASSIGNMENT_BIT_WIDTHS
    },
    "quantized_tensor_names": sorted(selected_names),
    "preserved_sensitive_tensor_names": sorted(PRESERVED_TENSOR_NAMES),
    "activations_quantized": False,
    "fp32_accuracy_percent": fp32_accuracy * 100,
    "accuracy_by_assignment_bits_percent": {
        str(int(row["assignment_bits"])): float(row["accuracy_percent"])
        for _, row in sweep_table.iterrows()
    },
    "reported_subbyte_storage_is_theoretical_packed_storage": True,
}, indent=2) + "\n", encoding="utf-8")
print(f"Saved table: {accuracy_path}")
print(f"Saved graph: {graph_path}")
print(f"Saved results: {results_path}")
